# Week 2 · pandas, matplotlib and scikit-learn on real data

This is the build part of the week: the same ideas as notebooks `04`–`05`, now with the tools everyone uses. The data is built into scikit-learn, so nothing needs downloading: the **breast cancer** dataset (569 tumours, 30 measurements each, malignant vs benign) and the **diabetes** dataset from [notebook 04 ↗](04_ml_intro_linear_regression.ipynb). **Reading:** McKinney Ch. 5 (pandas basics) and Ch. 9 (plotting); the scikit-learn "Getting Started" page.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **DataFrame**: pandas' table: rows are samples, columns are named features. Think of a NumPy 2-D array with column names.
- **Series**: One column of a DataFrame: a 1-D array with an index.
- **estimator**: Any scikit-learn model. They all have the same methods: `.fit(X, y)`, then `.predict(X)` (and `.predict_proba(X)` for classifiers).
- **pipeline**: A chain of steps (e.g. scaler → model) that behaves like one estimator, so every step is fitted on the training data only.

## 1. pandas: a table with column names

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
df = data.frame                 # a DataFrame: 30 feature columns + "target"
print(df.shape)                 # → (569, 31)
print(df.iloc[:3, :4])          # .iloc[rows, cols] by position, just like NumPy slicing
# →    mean radius  mean texture  mean perimeter  mean area
#   0        17.99         10.38           122.8     1001.0
#   1        20.57         17.77           132.9     1326.0
#   2        19.69         21.25           130.0     1203.0

Inside a notebook, writing `df.head()` as the last line of a cell shows a nicely formatted table of the first rows. Try it in the next cell.

In [ ]:
df.head()

**Selecting columns, and the shape rule from [notebook 03 ↗](03_shapes_and_slicing.ipynb) again:** one name in brackets gives a 1-D **Series**; a list of names gives a 2-D **DataFrame**.

```text
df["mean radius"]           one name   → a Series (1-D)          shape (569,)
     17.99  20.57  19.69  11.42  20.29  ...

df[["mean radius"]]         a LIST     → a DataFrame (2-D)       shape (569, 1)
        mean radius
   0         17.99
   1         20.57
   2         19.69
   ...
```

It's the same rule as `a[:, 0]` vs `a[:, [0]]` in [notebook 03 ↗](03_shapes_and_slicing.ipynb): one name drops the column axis, a list of names keeps it.

In [ ]:
print(df["mean radius"].shape, type(df["mean radius"]).__name__)        # → (569,) Series
print(df[["mean radius"]].shape, type(df[["mean radius"]]).__name__)    # → (569, 1) DataFrame

**The target.** Here `1` means benign and `0` means malignant, which is the opposite of what you might guess. Always check what the labels mean:

In [ ]:
print(data.target_names)              # → ['malignant' 'benign']   index 0, index 1
print(df["target"].value_counts())     # how many of each class
# → target
#   1    357
#   0    212
#   Name: count, dtype: int64

**Filtering rows with a mask** ([Week 1 · NumPy §6 ↗](../week1/02_numpy_lesson.ipynb): a True/False array used as an index keeps only the True positions; now with column names), **grouping**, and **checking for missing values**:

`df[df["mean radius"] > 20]` happens in two steps. On the first five rows:

```text
                 mean radius    > 20 ?
row 0               17.99       False     dropped
row 1               20.57       True      kept
row 2               19.69       False     dropped
row 3               11.42       False     dropped
row 4               20.29       True      kept

Step 1: df["mean radius"] > 20      → a True/False Series, one value per row
Step 2: df[ that Series ]           → keeps the rows where it's True (all 31 columns)
```

On the full table, 45 rows are True, which is why `big.shape` is `(45, 31)`.

In [ ]:
big = df[df["mean radius"] > 20]                  # rows where the condition is True
print(big.shape)                                    # → (45, 31)
print(df.groupby("target")["mean radius"].mean().round(2))   # average radius per class
# → target
#   0    17.46
#   1    12.15
#   Name: mean radius, dtype: float64
print(df.isna().sum().sum())                        # → 0   no missing values anywhere

**What `groupby` does: split, apply, combine**

On a small table of parts and their defect counts, `t.groupby("machine")["defects"].mean()`:

```text
machine  defects         SPLIT by machine          APPLY mean          COMBINE
   A        2            A:  2, 4, 0         →     (2+4+0)/3 = 2.0      machine
   B        5            B:  5, 3            →     (5+3)/2   = 4.0      A    2.0
   A        4            C:  1               →      1/1      = 1.0      B    4.0
   C        1                                                           C    1.0
   B        3
   A        0
```

In [ ]:
t = pd.DataFrame({"machine": ["A", "B", "A", "C", "B", "A"], "defects": [2, 5, 4, 1, 3, 0]})
print(t.groupby("machine")["defects"].mean())
# → machine
#   A    2.0
#   B    4.0
#   C    1.0
#   Name: defects, dtype: float64

In [ ]:
print(df[["mean radius", "mean texture"]].describe().round(2))    # summary statistics per column
# →        mean radius  mean texture
#   count       569.00        569.00
#   mean         14.13         19.29
#   std           3.52          4.30
#   min           6.98          9.71
#   25%          11.70         16.17
#   50%          13.37         18.84
#   75%          15.78         21.80
#   max          28.11         39.28

Malignant tumours (target 0) have a bigger average radius, 17.46 vs 12.15, so radius alone should already separate the classes a little. A plot shows how much.

## 2. matplotlib: look before you model

In [ ]:
import matplotlib.pyplot as plt
for label, name in [(0, "malignant"), (1, "benign")]:
    plt.hist(df.loc[df["target"] == label, "mean radius"], bins=30, alpha=0.6, label=name)
plt.xlabel("mean radius"); plt.ylabel("count"); plt.legend(); plt.title("One feature, two classes"); plt.show()

In [ ]:
plt.scatter(df["mean radius"], df["mean texture"], c=df["target"], cmap="coolwarm", s=10)
plt.xlabel("mean radius"); plt.ylabel("mean texture"); plt.title("Two features, colored by class"); plt.show()

The two classes overlap in the middle: no single line separates them perfectly. That's normal, and it's why the test accuracy below isn't 100%.

## 3. The scikit-learn workflow

**Every scikit-learn model follows the same three steps: split the data, `fit` on the training part, `predict` on the test part.**

In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop(columns="target").to_numpy()      # (569, 30) NumPy array of features
y = df["target"].to_numpy()                   # (569,)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(X_tr.shape, X_te.shape)                 # → (426, 30) (143, 30)
print(y_tr.mean().round(3), y_te.mean().round(3))   # → 0.627 0.629   stratify keeps the class mix the same

**A pipeline: scale, then classify.** `make_pipeline` chains the steps. `.fit` fits the scaler on `X_tr` only and passes the scaled data to the model, so no test information leaks into training (the leakage mistake from [notebook 05 §4 ↗](05_logistic_regression_knn.ipynb)).

```text
model.fit(X_tr, y_tr)
   X_tr ──► StandardScaler.fit(X_tr)        learns the mean and std of the TRAINING data only
        ──► .transform(X_tr)  ──► Z_tr      (X_tr - mean) / std
        ──► LogisticRegression.fit(Z_tr, y_tr)

model.predict(X_te)
   X_te ──► .transform(X_te)  ──► Z_te      uses the SAME training mean and std (no refit!)
        ──► LogisticRegression.predict(Z_te) ──► labels
```

The test data never touches `fit`. That's the leakage rule from [notebook 05 §4 ↗](05_logistic_regression_knn.ipynb), enforced automatically, and it's exactly what exercise 12 asks you to do by hand.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_tr, y_tr)
print(model.predict_proba(X_te[:3]).round(3))   # columns: P(class 0), P(class 1)
# → [[0.004 0.996]
#    [1.    0.   ]
#    [1.    0.   ]]
print(model.predict(X_te[:3]), y_te[:3])        # → [1 0 0] [1 0 0]
print(round(model.score(X_te, y_te), 3))        # → 0.958   .score() = accuracy for classifiers

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
pred = model.predict(X_te)
print(confusion_matrix(y_te, pred))      # rows = actual (0, 1), columns = predicted (0, 1)
# → [[50  3]
#    [ 3 87]]
print(round(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]), 4))   # → 0.9952
print(classification_report(y_te, pred, target_names=data.target_names, digits=3))

Read the confusion matrix with [notebook 05 §4 ↗](05_logistic_regression_knn.ipynb): of 53 malignant tumours, 50 were caught and 3 were missed; of 90 benign ones, 87 were right and 3 were false alarms. In medicine the 3 missed malignant cases matter most, so you would probably lower the threshold for "malignant" and accept more false alarms.

## 4. Cross-validation and choosing $k$ for kNN

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_tr, y_tr, cv=5)
print(scores.round(3), round(scores.mean(), 3))      # → [0.953 0.976 0.965 0.976 1.   ] 0.974

Five folds give five accuracies between 0.953 and 1.0. That spread is why one split alone isn't trustworthy. **Use CV on the training set to choose $k$**, and leave the test set alone:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
for k in [1, 3, 5, 11, 21, 51]:
    s = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)), X_tr, y_tr, cv=5).mean()
    print(k, round(s, 3))
# → 1 0.955
#   3 0.974
#   5 0.962
#   11 0.963
#   21 0.956
#   51 0.944

$k = 1$ overfits (0.955), very large $k$ oversmooths (0.944 at $k = 51$), and $k = 3$ does best here. Now the scaling lesson from [notebook 05 §5 ↗](05_logistic_regression_knn.ipynb), measured: the same kNN **without** the scaler:

In [ ]:
print(round(cross_val_score(KNeighborsClassifier(n_neighbors=5), X_tr, y_tr, cv=5).mean(), 3))   # → 0.939   vs 0.962 with scaling

## 5. Regression with scikit-learn

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
d = load_diabetes(as_frame=True)
Xa, Xb, ya, yb = train_test_split(d.data.to_numpy(), d.target.to_numpy(), test_size=0.25, random_state=0)
for m in [LinearRegression(), Ridge(alpha=0.1), Ridge(alpha=1.0)]:
    m.fit(Xa, ya)
    print(type(m).__name__, getattr(m, "alpha", ""), round(r2_score(yb, m.predict(Xb)), 3))
# → LinearRegression  0.359
#   Ridge 0.1 0.369
#   Ridge 1.0 0.357

A little ridge helps on the test set ($R^2$ from 0.359 to 0.369), and too much hurts again (0.357). That's the bias–variance trade-off from [Week 1 · math §8 ↗](../week1/03_math_essentials.ipynb), in a real model. In practice you'd choose `alpha` with cross-validation, exactly like $k$ above.

---
Next: `07_exercises.ipynb` (12 exercises: linear regression, ridge, gradient descent, logistic regression, metrics and kNN, all from scratch in NumPy). Then `08_sunday_quiz.ipynb`.